# 13.2 - Prompt Optimization

**Module 13 - Cost & Performance** | Netsetos GenAI Engineering

This notebook builds prompt-optimization tooling as a set of small runnable calculators - each cell prices one lever for cutting tokens without cutting quality: bloat that multiplies across every request, a per-component token analyzer, input trims, output cuts, automated compression, cache-aligned structure, and a measured A/B guard. Nothing here calls a model; every cell is arithmetic over illustrative token counts and prices so you can see exactly where the money goes.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This notebook makes no API calls - every cell is pure arithmetic over illustrative token counts and prices. This first cell is just environment prep and a note that the numbers throughout are fixed, pre-chosen values rather than anything sampled at runtime.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A setup cell, not a computation. It carries a commented-out `pip install anthropic` for the one illustrative pipeline later in the lesson, and a note that the values used throughout are seeded/fixed so the outputs are reproducible.

**How the code works - step by step**
- **Commented `pip install anthropic`** - uncomment only if you run the illustrative pipeline (the rest of the notebook needs no packages).
- **Reproducibility comment** - flags that the token counts and prices in later cells are fixed illustrative values, not random draws.

**In one line:** environment prep - no imports run, no output.

**What you'll see:** (no output - environment prep)

## 1 - Why bloat costs twice

Every token in a prompt costs money on input, and a system prompt runs on **every single request** - so an oversized system prompt is not a one-time waste, it is that waste multiplied across all traffic. This cell prices the gap between an 800-token and a 200-token system prompt at 10,000 requests a day, which is the highest-ROI framing in the whole lesson: trimming a high-traffic prompt beats a cheaper model, because it applies to every call forever.

In [ ]:
# Illustrative price (USD per 1M tokens): flash input $0.15. Every token costs money on INPUT,
# and a system prompt runs on EVERY request - so bloat multiplies across all of them.
IN_PRICE = 0.15
bloated, concise = 800, 200          # same-quality system prompts, in tokens
requests_per_day = 10000

waste_tok = (bloated - concise) * requests_per_day
waste_usd_day = waste_tok * IN_PRICE / 1_000_000
print("A system prompt runs on every request. Two versions, same output quality:")
print("  bloated system prompt: {} tokens".format(bloated))
print("  concise system prompt: {} tokens  ({}x smaller)".format(concise, bloated // concise))
print()
print("At {:,} requests/day the {}-token difference is paid EVERY time:".format(requests_per_day, bloated - concise))
print("  wasted input tokens/day: {:,}".format(waste_tok))
print("  wasted cost/day (flash): ${:.2f}   -> ${:.2f}/month".format(waste_usd_day, waste_usd_day * 30))
print()
print("Bloat costs twice: once as input you pay for, once in slower attention. Most prompts are 40-60% bloat.")
print("Trimming the system prompt is the highest-ROI cut in AI engineering - it applies to every call, forever.")

# Output:
# A system prompt runs on every request. Two versions, same output quality:
#   bloated system prompt: 800 tokens
#   concise system prompt: 200 tokens  (4x smaller)
#
# At 10,000 requests/day the 600-token difference is paid EVERY time:
#   wasted input tokens/day: 6,000,000
#   wasted cost/day (flash): $0.90   -> $27.00/month
#
# Bloat costs twice: once as input you pay for, once in slower attention. Most prompts are 40-60% bloat.
# Trimming the system prompt is the highest-ROI cut in AI engineering - it applies to every call, forever.

**Explanation**

A cost calculator, not a model call. It takes two same-quality system prompts of different sizes and multiplies the token gap by daily request volume to show that bloat is paid over and over, then converts that to daily and monthly dollars at an illustrative input price.

**How the code works - step by step**
- **`IN_PRICE`, `bloated`, `concise`, `requests_per_day`** - the inputs: $0.15 per 1M input tokens, an 800- vs 200-token system prompt, 10,000 requests/day.
- **`waste_tok`** - the per-request token gap (600) times requests/day = wasted tokens per day.
- **`waste_usd_day`** - converts wasted tokens to dollars, and prints the monthly figure as `* 30`.
- **The prints** - state that both prompts have the same output quality, then show the gap is paid on every request and what that costs.

**In one line:** (bloated - concise) tokens x every request = the daily waste bloat quietly bills you.

**What you'll see:** The 600-token difference becomes 6,000,000 wasted input tokens/day, costing $0.90/day and $27.00/month at the flash input price - with the closing lines that bloat costs twice and most prompts are 40-60% bloat.

## 2 - Find the waste: analyze by component

You cannot optimize what you have not measured. A prompt is not one blob - it is a stack of components (system prompt, few-shot examples, retrieved context, conversation history, user query), each contributing tokens. This cell counts the tokens in each and ranks them, so you target the biggest, most redundant component first instead of guessing.

In [ ]:
# You cannot optimize what you have not measured. Break the prompt into components and count tokens.
components = {  # (illustrative token counts)
    "system prompt":     480,
    "few-shot examples": 240,
    "retrieved context": 3000,
    "conversation history": 800,
    "user query":        50,
}
total = sum(components.values())
print("Prompt anatomy ({} tokens total):".format(total))
for name, tok in sorted(components.items(), key=lambda kv: -kv[1]):
    bar = "#" * round(tok / total * 40)
    print("  {:<20} {:>5} tok  {:>4.0f}%  {}".format(name, tok, tok / total * 100, bar))
print()
biggest = max(components, key=components.get)
print("Biggest component: {} at {:.0f}% - and the most structurally redundant, so it is the FIRST target.".format(biggest, components[biggest] / total * 100))
print("The user query is {:.0f}% of the prompt - a rounding error. The tokens (and the money) are in the context and the system prompt.".format(components["user query"] / total * 100))

# Output:
# Prompt anatomy (4570 tokens total):
#   retrieved context     3000 tok    66%  ##########################
#   conversation history   800 tok    18%  #######
#   system prompt          480 tok    11%  ####
#   few-shot examples      240 tok     5%  ##
#   user query              50 tok     1%  
#
# Biggest component: retrieved context at 66% - and the most structurally redundant, so it is the FIRST target.
# The user query is 1% of the prompt - a rounding error. The tokens (and the money) are in the context and the system prompt.

**Explanation**

A measurement harness that turns 'this prompt feels long' into a ranked breakdown. It sums an illustrative per-component token budget, sorts the components largest-first, and draws an ASCII bar for each so the retrieved context visibly dominates while the user query is a rounding error.

**How the code works - step by step**
- **`components` dict** - the five prompt parts with illustrative token counts (retrieved context is by far the largest at 3000).
- **`total`** - sums all component tokens.
- **The sorted loop** - orders components largest-first, printing each one's token count, its percentage of the total, and a `#` bar scaled to 40 characters.
- **`biggest` / final prints** - names the top component as the first target and calls out the query as a rounding error.

**In one line:** count tokens per component, rank them, cut the heaviest first.

**What you'll see:** A 4570-token anatomy where retrieved context is 66%, history 18%, system prompt 11%, few-shot 5%, and the user query just 1% - with the closing lines that the context is the first target and the query is a rounding error.

## 3 - Trim the input

The two manual moves that go furthest on the input side: rewrite verbose instructions as a concise single-purpose directive (roughly 30-50% off, and clearer), and prune few-shot examples down to the one to three that hold quality. This cell prices both trims on illustrative token counts and totals the input saving.

In [ ]:
# The manual, highest-ROI cut: concise instructions + few-shot pruning. (illustrative token counts)
verbose = 120   # "Please kindly ensure that you carefully..." with redundancy + filler
concise = 55    # a single-purpose directive, same meaning
print("Trim 1 - concise instructions:")
print("  verbose instruction: {} tokens".format(verbose))
print("  concise instruction: {} tokens   ({:.0f}% smaller, same meaning, LESS ambiguous)".format(concise, (1 - concise / verbose) * 100))
print()
# Few-shot pruning: more examples do not always mean better output.
per_example = 200
before, after = 3, 1
print("Trim 2 - few-shot pruning ({} tokens/example):".format(per_example))
print("  {} examples: {} tokens".format(before, before * per_example))
print("  {} example:  {} tokens   ({:.0f}% smaller; 1-3 examples usually match 5+ on quality)".format(after, after * per_example, (1 - after / before) * 100))
print()
input_before = verbose + before * per_example
input_after = concise + after * per_example
print("Input tokens: {} -> {}  = {:.0f}% off the instruction + examples, quality held.".format(input_before, input_after, (1 - input_after / input_before) * 100))
print("Also drop context the model already knows (common formats, well-known facts) - do not re-explain it.")

# Output:
# Trim 1 - concise instructions:
#   verbose instruction: 120 tokens
#   concise instruction: 55 tokens   (54% smaller, same meaning, LESS ambiguous)
#
# Trim 2 - few-shot pruning (200 tokens/example):
#   3 examples: 600 tokens
#   1 example:  200 tokens   (67% smaller; 1-3 examples usually match 5+ on quality)
#
# Input tokens: 720 -> 255  = 65% off the instruction + examples, quality held.
# Also drop context the model already knows (common formats, well-known facts) - do not re-explain it.

**Explanation**

A before/after token calculator for the two highest-ROI input trims. It compares a padded instruction against a concise one, then a three-example few-shot against a one-example one, and sums both to show roughly two-thirds off the instruction-plus-examples with quality held.

**How the code works - step by step**
- **`verbose`, `concise`** - a 120-token padded instruction vs a 55-token single-purpose directive; prints the percent saved.
- **`per_example`, `before`, `after`** - 200 tokens per example, pruning from 3 examples to 1, with the note that 1-3 usually match 5+ on quality.
- **`input_before` / `input_after`** - totals instruction + examples for each version and prints the combined percent off.
- **Final line** - reminds you to also drop context the model already knows (common formats, well-known facts).

**In one line:** concise instruction + fewer examples = about two-thirds off the input, same meaning.

**What you'll see:** The instruction drops 120 -> 55 tokens (54% smaller), the few-shot drops 600 -> 200 tokens (67% smaller), and the combined input falls 720 -> 255 tokens (65% off), plus the reminder to drop what the model already knows.

## 4 - Cut the output

The lever most people miss: output tokens are priced several times higher than input, so trimming the output saves more per token than the same cut on the input. This cell prices free-form prose against a structured answer of the same information, then proves that an identical-size cut on output beats it on input by the price ratio.

In [ ]:
# Output is the BIGGER lever: output tokens cost more per token, so trimming output beats trimming input.
IN_PRICE, OUT_PRICE = 0.15, 0.60     # flash illustrative; output is 4x input per token
prose_out, structured_out = 300, 150 # same information, free-form prose vs a structured answer

print("Same answer, two formats:")
print("  free-form prose: {} output tokens".format(prose_out))
print("  structured (bullets/JSON): {} output tokens   ({:.0f}% fewer - the model stops padding with prose)".format(structured_out, (1 - structured_out / prose_out) * 100))
print()
saved_out = (prose_out - structured_out) * OUT_PRICE / 1_000_000
saved_in_same = (prose_out - structured_out) * IN_PRICE / 1_000_000
print("Trimming {} OUTPUT tokens saves ${:.6f}".format(prose_out - structured_out, saved_out))
print("The SAME {}-token cut on INPUT would save only ${:.6f} - {:.0f}x less, because output is priced {:.0f}x higher.".format(prose_out - structured_out, saved_in_same, saved_out / saved_in_same, OUT_PRICE / IN_PRICE))
print()
print("So: request structured output, budget the response ('answer in 3 short bullets'), and set max_tokens.")
print("Concise input + controlled output stacks to ~50-80% total savings.")

# Output:
# Same answer, two formats:
#   free-form prose: 300 output tokens
#   structured (bullets/JSON): 150 output tokens   (50% fewer - the model stops padding with prose)
#
# Trimming 150 OUTPUT tokens saves $0.000090
# The SAME 150-token cut on INPUT would save only $0.000023 - 4x less, because output is priced 4x higher.
#
# So: request structured output, budget the response ('answer in 3 short bullets'), and set max_tokens.
# Concise input + controlled output stacks to ~50-80% total savings.

**Explanation**

A per-token cost comparison that makes the output-vs-input asymmetry concrete. It halves the output token count by switching prose to a structured format, then costs the same token cut on both output and input to show output cuts save 4x more at the illustrative prices.

**How the code works - step by step**
- **`IN_PRICE`, `OUT_PRICE`** - $0.15 input vs $0.60 output per 1M tokens (output is 4x per token here).
- **`prose_out`, `structured_out`** - the same answer as 300 tokens of prose vs 150 tokens of structured output (50% fewer).
- **`saved_out` vs `saved_in_same`** - costs the identical 150-token cut on output and on input, and prints the ratio between them.
- **Final prints** - the operating advice: request structured output, budget the response length, set `max_tokens`, and stack it with concise input for ~50-80% total.

**In one line:** same-size cut, but on the 4x-priced output it saves 4x more money.

**What you'll see:** Structured output is 50% fewer tokens (300 -> 150); trimming those 150 output tokens saves $0.000090 while the same input cut saves only $0.000023 - 4x less - plus the advice to budget output and cap max_tokens.

## 5 - Automated compression: LLMLingua

Some prompts are too long to hand-edit - a 2000-token retrieved context, a long history. Automated compressors (LLMLingua) use a small model to score each token's informativeness and drop the low-value ones, with a budget controller applying different keep-ratios per component. This cell models that squeeze on illustrative counts and prices the resulting input saving.

In [ ]:
# When a prompt is too long to hand-trim, a compressor (LLMLingua) scores each token's
# informativeness with a small model and drops the low-value ones. A BUDGET CONTROLLER keeps
# different ratios per component (instructions high, demonstrations low).
IN_PRICE = 0.15
parts = {  # (tokens, keep_ratio) - the budget controller keeps more of the instruction, less of demos
    "instruction":    (240, 0.25),
    "demonstrations": (480, 0.04),
    "question":       (127, 0.12),
}
orig = sum(t for t, _ in parts.values())
kept = sum(round(t * r) for t, r in parts.values())
print("LLMLingua-style compression (keep the informative tokens, drop the filler):")
for name, (t, r) in parts.items():
    print("  {:<15} {:>4} tok -> keep {:>2.0f}% -> {:>3} tok".format(name, t, r * 100, round(t * r)))
print()
ratio = orig / kept
print("Total: {} -> {} tokens = {:.1f}x compression".format(orig, kept, ratio))
print("  input cost drops the same {:.1f}x: ${:.6f} -> ${:.6f}".format(ratio, orig * IN_PRICE / 1_000_000, kept * IN_PRICE / 1_000_000))
print("  measured quality loss: under 2% (illustrative) - compression is lossy, so you VALIDATE it (Step 7).")
print("Use it for the parts you cannot hand-trim: a big retrieved context, a long history.")

# Output:
# LLMLingua-style compression (keep the informative tokens, drop the filler):
#   instruction      240 tok -> keep 25% ->  60 tok
#   demonstrations   480 tok -> keep  4% ->  19 tok
#   question         127 tok -> keep 12% ->  15 tok
#
# Total: 847 -> 94 tokens = 9.0x compression
#   input cost drops the same 9.0x: $0.000127 -> $0.000014
#   measured quality loss: under 2% (illustrative) - compression is lossy, so you VALIDATE it (Step 7).
# Use it for the parts you cannot hand-trim: a big retrieved context, a long history.

**Explanation**

A compression-ratio calculator modelling LLMLingua's budget controller. It keeps a different fraction of each component (a lot of the instruction, almost none of the demonstrations), sums the survivors, and reports the overall compression factor and matching input-cost drop - with the reminder that compression is lossy and must be validated.

**How the code works - step by step**
- **`parts` dict** - each component as `(tokens, keep_ratio)`: instruction keeps 25%, demonstrations 4%, question 12%.
- **`orig` / `kept`** - total tokens before, and the rounded sum of the kept fractions after.
- **The per-part loop** - prints each component's original tokens, keep percent, and surviving tokens.
- **`ratio`** - orig/kept as the compression factor, with the same factor applied to input cost.
- **Final prints** - the sub-2% quality-loss claim and the reminder to validate (Step 7) and use it only for un-hand-trimmable parts.

**In one line:** score tokens by informativeness, keep the signal per a per-component budget, ~9x smaller.

**What you'll see:** The prompt collapses 847 -> 94 tokens (9.0x compression), input cost drops the same 9.0x ($0.000127 -> $0.000014), at an illustrative under-2% quality loss - flagged as lossy and to be validated.

## 6 - Structure for the cache

A prompt's shape matters as much as its length. Prompt caching charges the cached prefix at roughly a tenth of the input price - but only when that prefix is identical and comes first. This cell prices two layouts of the same prompt: stable prefix first (cache hit) vs variable content first (cache miss), to show the structural rule.

In [ ]:
# Prompt caching (12.4) charges the cached prefix at ~10% of input price - but ONLY if it is
# IDENTICAL and comes FIRST. So structure the prompt: stable prefix, then dynamic suffix.
IN_PRICE = 0.15
CACHED_PRICE = IN_PRICE * 0.10       # cached input ~10% of input
stable = 3500     # system prompt + few-shot examples - never change
dynamic = 100     # the user query - changes every call

# Layout A: stable prefix FIRST (cacheable) + dynamic suffix (fresh)
hit = (stable * CACHED_PRICE + dynamic * IN_PRICE) / 1_000_000
# Layout B: something variable early -> the prefix is not identical -> CACHE MISS (all fresh)
miss = (stable + dynamic) * IN_PRICE / 1_000_000

print("A prompt = a {}-token STABLE prefix (system + few-shot) + a {}-token DYNAMIC suffix (query):".format(stable, dynamic))
print("  stable-prefix-FIRST (cache HIT):  ${:.6f}   (the {}-token prefix is cached at 10% of input)".format(hit, stable))
print("  variable-content-first (cache MISS): ${:.6f}   (prefix not identical -> nothing cached, all fresh)".format(miss))
print()
print("  structuring the prompt saves ${:.6f} PER REQUEST - about {:.1f}x cheaper on the repeated prefix.".format(miss - hit, miss / hit))
print("Put anything variable early and you cache-miss every call. Optimizing the prompt's SHAPE matters as much as its length.")

# Output:
# A prompt = a 3500-token STABLE prefix (system + few-shot) + a 100-token DYNAMIC suffix (query):
#   stable-prefix-FIRST (cache HIT):  $0.000068   (the 3500-token prefix is cached at 10% of input)
#   variable-content-first (cache MISS): $0.000540   (prefix not identical -> nothing cached, all fresh)
#
#   structuring the prompt saves $0.000472 PER REQUEST - about 8.0x cheaper on the repeated prefix.
# Put anything variable early and you cache-miss every call. Optimizing the prompt's SHAPE matters as much as its length.

**Explanation**

A caching cost comparison of two prompt layouts. It splits the prompt into a large stable prefix and a small dynamic suffix, then prices the cache-hit layout (prefix billed at 10%) against the cache-miss layout (nothing cached, all fresh) to show that shape alone changes the bill several-fold.

**How the code works - step by step**
- **`IN_PRICE`, `CACHED_PRICE`** - full input price and the cached price at 10% of it.
- **`stable`, `dynamic`** - a 3500-token stable prefix (system + few-shot) and a 100-token dynamic suffix (query).
- **`hit`** - stable prefix billed at the cached rate + dynamic at full rate = the stable-prefix-first layout.
- **`miss`** - the whole prompt at full rate, because variable content up front breaks the identical-prefix requirement.
- **Final prints** - the per-request saving and the ratio, and the rule that anything variable early cache-misses every call.

**In one line:** stable block first = cached prefix at 10%; variable first = pay full price every call.

**What you'll see:** The cache-hit layout costs $0.000068 vs the cache-miss layout at $0.000540 - about 8x cheaper on the repeated prefix, saving $0.000472 per request - with the rule to keep variable content out of the leading prefix.

## 7 - Measure the tradeoff

Never ship a cut on faith. Every optimization so far risks quality, so the final discipline is to A/B the optimized prompt against the original on a fixed eval set and keep it only if quality holds - plus a token-budget guardrail so bloat cannot creep back. This cell runs that A/B and the budget check on illustrative scores.

In [ ]:
# Never ship a cut on faith. A/B the optimized prompt against the original on an EVAL SET,
# and keep it ONLY if quality holds. Plus a token-budget guardrail so bloat cannot creep back.
QUALITY_FLOOR = 8.0
original = {"tokens": 800, "quality": 8.6}
candidates = {
    "concise + structured":     {"tokens": 350, "quality": 8.5},   # big cut, quality holds
    "aggressive compression":   {"tokens": 180, "quality": 7.2},   # too lossy
}
print("A/B each optimization vs the original (original: {} tokens, quality {}/10):".format(original["tokens"], original["quality"]))
for name, c in candidates.items():
    saved = (1 - c["tokens"] / original["tokens"]) * 100
    keep = c["quality"] >= QUALITY_FLOOR and c["quality"] >= original["quality"] - 0.3
    verdict = "KEEP" if keep else "REJECT (quality below the {} floor / tolerance)".format(QUALITY_FLOOR)
    print("  {:<24} {} tok ({:.0f}% off), quality {} -> {}".format(name, c["tokens"], saved, c["quality"], verdict))
print()
# Token-budget guardrail in the template: reject an over-budget assembly.
BUDGET = 500
for assembled in [420, 620]:
    ok = assembled <= BUDGET
    print("  token-budget guardrail: assembled prompt {} tokens vs {} budget -> {}".format(assembled, BUDGET, "OK" if ok else "REJECT (over budget)"))
print()
print("Cost down, quality held - MEASURED, not assumed. Keep the win only when the eval score survives the cut.")

# Output:
# A/B each optimization vs the original (original: 800 tokens, quality 8.6/10):
#   concise + structured     350 tok (56% off), quality 8.5 -> KEEP
#   aggressive compression   180 tok (78% off), quality 7.2 -> REJECT (quality below the 8.0 floor / tolerance)
#
#   token-budget guardrail: assembled prompt 420 tokens vs 500 budget -> OK
#   token-budget guardrail: assembled prompt 620 tokens vs 500 budget -> REJECT (over budget)
#
# Cost down, quality held - MEASURED, not assumed. Keep the win only when the eval score survives the cut.

**Explanation**

A decision harness, not a computation of savings. It scores each candidate optimization against a quality floor and a tolerance around the original's score, printing KEEP or REJECT, then runs a template token-budget assertion so an over-budget assembly is rejected. It encodes the whole lesson's rule: cost down only when quality holds.

**How the code works - step by step**
- **`QUALITY_FLOOR`, `original`** - an 8.0 floor and the original prompt at 800 tokens, quality 8.6.
- **`candidates` dict** - two optimizations: 'concise + structured' (350 tok, 8.5) and 'aggressive compression' (180 tok, 7.2).
- **The A/B loop** - for each, computes percent saved and a `keep` flag (must clear the floor AND be within 0.3 of the original), printing KEEP or REJECT.
- **Budget guardrail loop** - checks assembled sizes 420 and 620 against a 500-token budget, rejecting the over-budget one.
- **Final line** - the lesson in one phrase: cost down, quality held, measured not assumed.

**In one line:** A/B on the eval set, keep only the cut that holds quality, and cap the token budget.

**What you'll see:** 'concise + structured' is KEEP (56% off, quality 8.5); 'aggressive compression' is REJECT (78% off but 7.2, below the floor); the 420-token assembly passes the 500 budget and the 620-token one is rejected.

Put together, these seven calculators are the full prompt-optimization pipeline: bloat costs twice and runs on every request (1), so you measure by component to find where the tokens hide (2), trim the input (3), cut the more expensive output (4), compress the un-hand-trimmable contexts (5), structure the prompt stable-prefix-first so the repeated part caches (6), and A/B every cut on an eval set with a token-budget guard so you keep the win only when quality holds (7). Every cell is deliberately keyless arithmetic - the real levers plug into an eval set (Lesson 14.4) and caching mechanics (Lesson 12.4). Next in Module 13: model routing, distillation, and serving economics - the cost levers beyond the prompt itself.